# Module 4 - VSCMG Dynamics

Variable-Speed Control Moment Gyroscopes (VSCMGs) combine the operating principles of **reaction wheels (RWs)** and **control moment gyroscopes (CMGs)**. By controlling both rotor spin speed and gimbal motion, a VSCMG can exchange angular momentum with the spacecraft through two independent actuator motions.

The basic mechanics of momentum-exchange devices were developed in the first *Spacecraft Dynamics and Control* specialization. This module builds on that foundation by revisiting the single-VSCMG model compactly, then extending it to the numerical modeling of spacecraft carrying multiple VSCMGs.

The development follows the progression

$$
\boxed{
\text{geometry and moving frames}
\;\rightarrow\;
\text{inertia and angular momentum}
\;\rightarrow\;
\text{equations of motion}
\;\rightarrow\;
\text{motor torques}
\;\rightarrow\;
\text{numerical simulation}
}
$$

The single-VSCMG model is developed first from its geometry and momentum relations. Reaction-wheel and CMG behavior then appear naturally as special cases of the general VSCMG formulation. The same device-level model is subsequently extended to multiple VSCMGs, with particular attention given to numerical implementation, simulation, and verification using the work-energy principle.

This notebook therefore focuses on **reconstructing, implementing, and verifying** the dynamics rather than repeating the full derivations developed previously.

> **Prior material:**  
> [Specialization 1, Course 2, Module 4 - Equations of Motion with Momentum Exchange Devices](https://github.com/johnm3398/Spacecraft-Dynamics-and-Control/blob/main/01_spacecraft_dynamics_and_control_specialization/02_kinetics/Module%204%20-%20Equations%20of%20Motion%20with%20Momentum%20Exchange%20Devices.ipynb)  
> [Specialization 1, Course 2 - Momentum Exchange Devices Slides](https://github.com/johnm3398/Spacecraft-Dynamics-and-Control/blob/main/01_spacecraft_dynamics_and_control_specialization/02_kinetics/Slides/5%20-%20Momentum%20Exchange%20Devices.pdf)

In [1]:
import numpy as np

from scipy.integrate import solve_ivp

import matplotlib.pyplot as plt

import sys
sys.path.insert(0, r"../../")
import AttitudeKinematicsLib as ak

# 1 - Single VSCMG Modeling

## 1.1 - Momentum Exchange Devices and the VSCMG

A **momentum exchange device** controls spacecraft rotation by exchanging angular momentum between an internal rotating assembly and the spacecraft body.

In the absence of external torque, the total angular momentum of the spacecraft-device system is conserved. The actuator therefore does not create angular momentum; instead, an internal motor changes the momentum stored in the device and the spacecraft body responds with an equal and opposite change.

Conceptually,

$$
\boxed{
\text{change device angular momentum}
\;\Longleftrightarrow\;
\text{change spacecraft angular momentum}
}
$$

The three devices of interest differ primarily in **how the internal angular momentum is changed**:

- A **reaction wheel (RW)** has a fixed spin axis. Control is generated by changing the rotor spin rate and therefore the magnitude of its angular momentum.
- A **control moment gyroscope (CMG)** typically maintains a large rotor spin rate and generates control torque by gimbaling the rotor, thereby changing the **direction** of its angular momentum.
- A **variable-speed control moment gyroscope (VSCMG)** allows both the rotor speed and gimbal angle to vary.

A VSCMG therefore combines the two actuation mechanisms:

$$
\boxed{
\dot{\Omega}
\;\rightarrow\;
\text{change momentum magnitude}
}
$$

and

$$
\boxed{
\dot{\gamma}
\;\rightarrow\;
\text{change momentum direction}.
}
$$

This makes the VSCMG the most general of the three models. Reaction-wheel and conventional CMG behavior can later be recovered as special cases by restricting one of these internal motions.

The first step in constructing its equations of motion is therefore to define the geometry of the rotating assembly and the moving basis vectors attached to it.

## 1.2 - VSCMG Geometry and Moving Frames

A VSCMG has two internal motions going on at the same time: the rotor spins about its own axis, while the gimbal rotates that whole rotor assembly about a second axis.

It is easiest to keep track of all of this using the three orthogonal directions shown below.

<img src="Images/VSCMG_Frame.PNG" alt="VSCMG Frame" width="500" align="left" style="margin-right: 20px;"/>

- <ins>**Spin axis**</ins> ($\hat{\mathbf g}_s$)  
  The axis the rotor actually spins about. The rotor spin rate about this axis is $\Omega$.

- <ins>**Gimbal axis**</ins> ($\hat{\mathbf g}_g$)  
  The axis about which the entire rotor-gimbal assembly rotates. This axis is fixed relative to the spacecraft body frame $\mathcal{B}$.

- <ins>**Transverse axis**</ins> ($\hat{\mathbf g}_t$)  
  The third direction needed to complete the right-handed orthonormal triad,

  $$
  \hat{\mathbf g}_t = \hat{\mathbf g}_g \times \hat{\mathbf g}_s.
  $$

- <ins>**Rotor spin rate**</ins> ($\Omega$)  
  The angular velocity of the rotor relative to the gimbal frame about $\hat{\mathbf g}_s$.

- <ins>**Gimbal angle**</ins> ($\gamma$)  
  Describes how far the gimbal has rotated about $\hat{\mathbf g}_g$.

- <ins>**Gimbal rate**</ins> ($\dot{\gamma}$)  
  The angular velocity of the gimbal frame relative to the spacecraft body.

- <ins>**Gimbal frame**</ins> ($\mathcal{G}$)

  $$
  \mathcal{G} =
  \left\{
  \hat{\mathbf g}_s,\,
  \hat{\mathbf g}_t,\,
  \hat{\mathbf g}_g
  \right\}.
  $$

  These three unit vectors define the moving coordinate frame attached to the gimbal assembly.

<br clear="left"/>

The key thing to notice is that $\hat{\mathbf g}_g$ stays fixed in the spacecraft body, while $\hat{\mathbf g}_s$ and $\hat{\mathbf g}_t$ rotate around it as the gimbal moves.

The angular velocity of the gimbal frame relative to the spacecraft body is therefore

$$
\boxed{
\boldsymbol{\omega}_{G/B} = \dot{\gamma}\, \hat{\mathbf g}_g.
}
$$

The rotor then has another rotation on top of this: it spins relative to the gimbal frame about $\hat{\mathbf g}_s$,

$$
\boxed{
\boldsymbol{\omega}_{W/G} = \Omega\, \hat{\mathbf g}_s.
}
$$

So there are really two nested rotations taking place:

$$
\boxed{
\mathcal{B}
\;\xrightarrow{\;\dot{\gamma}\hat{\mathbf g}_g\;}
\mathcal{G}
\;\xrightarrow{\;\Omega\hat{\mathbf g}_s\;}
\mathcal{W}.
}
$$

Here, $\mathcal{B}$ is the spacecraft body frame, $\mathcal{G}$ is the gimbal frame, and $\mathcal{W}$ is the spinning rotor frame.

**<ins>How the Gimbal-Frame Axes Move</ins>**

The vectors $\hat{\mathbf g}_s$, $\hat{\mathbf g}_t$, and $\hat{\mathbf g}_g$ are fixed when viewed from $\mathcal G$ itself. From the spacecraft body, though, the spin and transverse axes turn as the gimbal moves. Let's write that motion down before taking any derivatives.

Choose a reference time $t_0$, and let $\gamma_0=\gamma(t_0)$. The change in gimbal angle is $\Delta\gamma=\gamma(t)-\gamma_0$. With every axis written in **body-frame components**,

$$
\begin{aligned}
\hat{\mathbf g}_s(t) &= \cos(\Delta\gamma)\hat{\mathbf g}_s(t_0)
    + \sin(\Delta\gamma)\hat{\mathbf g}_t(t_0), \\
\hat{\mathbf g}_t(t) &= -\sin(\Delta\gamma)\hat{\mathbf g}_s(t_0)
    + \cos(\Delta\gamma)\hat{\mathbf g}_t(t_0), \\
\hat{\mathbf g}_g(t) &= \hat{\mathbf g}_g(t_0).
\end{aligned}
$$

The initial columns on the right are constant body-coordinate directions. These equations describe the axes turning inside the spacecraft; they do not yet include the spacecraft's own rotation relative to inertial space.

**Differentiate the geometry in the body frame.** Since $d(\Delta\gamma)/dt=\dot{\gamma}$,

$$
\begin{aligned}
{}^B\frac{d\hat{\mathbf g}_s}{dt}
&= \dot{\gamma}\left[-\sin(\Delta\gamma)\hat{\mathbf g}_s(t_0)
    + \cos(\Delta\gamma)\hat{\mathbf g}_t(t_0)\right] \\
&= \dot{\gamma}\hat{\mathbf g}_t,
\end{aligned}
$$

and

$$
\begin{aligned}
{}^B\frac{d\hat{\mathbf g}_t}{dt}
&= -\dot{\gamma}\left[\cos(\Delta\gamma)\hat{\mathbf g}_s(t_0)
    + \sin(\Delta\gamma)\hat{\mathbf g}_t(t_0)\right] \\
&= -\dot{\gamma}\hat{\mathbf g}_s.
\end{aligned}
$$

The gimbal axis stays fixed in the body, so ${}^B d\hat{\mathbf g}_g/dt=\mathbf 0$.

**The same result from the transport theorem.** For any gimbal-fixed basis vector, indexed by $i=s,t,g$,

$$
\begin{aligned}
{}^B\frac{d\hat{\mathbf g}_i}{dt}
&= {}^G\frac{d\hat{\mathbf g}_i}{dt}
    + \boldsymbol{\omega}_{G/B}\times\hat{\mathbf g}_i \\
&= \mathbf 0 + \dot{\gamma}\hat{\mathbf g}_g\times\hat{\mathbf g}_i.
\end{aligned}
$$

The signs follow from the right-handed triad:

$$
\begin{aligned}
\hat{\mathbf g}_g\times\hat{\mathbf g}_s &= \hat{\mathbf g}_t, \\
\hat{\mathbf g}_g\times\hat{\mathbf g}_t &= -\hat{\mathbf g}_s, \\
\hat{\mathbf g}_g\times\hat{\mathbf g}_g &= \mathbf 0.
\end{aligned}
$$

So the three body-frame rates are

$$
\boxed{
\begin{aligned}
{}^B\frac{d\hat{\mathbf g}_s}{dt} &= \dot{\gamma}\hat{\mathbf g}_t, \\
{}^B\frac{d\hat{\mathbf g}_t}{dt} &= -\dot{\gamma}\hat{\mathbf g}_s, \\
{}^B\frac{d\hat{\mathbf g}_g}{dt} &= \mathbf 0.
\end{aligned}
}
$$

This is the geometry behind CMG action: gimbaling turns the spin axis toward the transverse axis. Later, when that axis carries wheel angular momentum, its motion will produce a transverse momentum-rate term. First we need to work out exactly how much momentum each part carries.


## 1.3 - VSCMG Inertia and System Angular Momentum

With the geometry sorted out, the next step is to build the angular momentum of the full spacecraft-device system:

$$
\mathbf H = \mathbf H_B + \mathbf H_G + \mathbf H_W.
$$

Here $\mathbf H_B$ is the spacecraft-body contribution, $\mathbf H_G$ is the gimbal contribution, and $\mathbf H_W$ is the wheel contribution. Once we have all three, the dynamics follow from

$$
{}^N\frac{d\mathbf H}{dt} = \mathbf L,
$$

where $\mathcal N$ is the inertial frame and $\mathbf L$ is the net external torque about the complete system's center of mass.

We use the balanced-device model underlying the reference derivation. The device centers stay fixed in the body, and the separately modeled gimbal and wheel inertias describe rotation about their own centers. Any constant contributions from their offsets are included once in the spacecraft inertia $[I_s]$. We'll keep that bookkeeping explicit when we write $\mathbf H_B$.

For now, there are two jobs: express the inertias in a common frame, then apply each inertia to the **inertial angular velocity of that part**.


**<ins>Gimbal and Wheel Inertias</ins>**

The gimbal axes are principal axes of the gimbal assembly, so its inertia matrix in $\mathcal G$ is

$$
{}^G[I_G] =
\begin{bmatrix}
I_{G_s} & 0 & 0 \\
0 & I_{G_t} & 0 \\
0 & 0 & I_{G_g}
\end{bmatrix}.
$$

The wheel is axisymmetric about its spin axis. Its spin moment is $I_{W_s}$, and both transverse moments are $I_{W_t}$:

$$
{}^W[I_W] =
\begin{bmatrix}
I_{W_s} & 0 & 0 \\
0 & I_{W_t} & 0 \\
0 & 0 & I_{W_t}
\end{bmatrix}.
$$

That repeated transverse moment is useful: rotating the wheel about its spin axis leaves its inertia unchanged. We can see this directly from the transformation between $\mathcal W$ and $\mathcal G$.

Let $\theta$ be the wheel's positive spin angle relative to the gimbal, with $\dot{\theta}=\Omega$. The direction cosine matrix mapping wheel components to gimbal components is

$$
[GW] =
\begin{bmatrix}
1 & 0 & 0 \\
0 & \cos\theta & -\sin\theta \\
0 & \sin\theta & \cos\theta
\end{bmatrix}.
$$

Apply the tensor transformation:

$$
\begin{aligned}
{}^G[I_W] &= [GW]\,{}^W[I_W]\,[GW]^T \\
&=
\begin{bmatrix}
I_{W_s} & 0 & 0 \\
0 & I_{W_t}(\cos^2\theta+\sin^2\theta)
  & I_{W_t}(\cos\theta\sin\theta-\sin\theta\cos\theta) \\
0 & I_{W_t}(\sin\theta\cos\theta-\cos\theta\sin\theta)
  & I_{W_t}(\sin^2\theta+\cos^2\theta)
\end{bmatrix} \\
&=
\begin{bmatrix}
I_{W_s} & 0 & 0 \\
0 & I_{W_t} & 0 \\
0 & 0 & I_{W_t}
\end{bmatrix}.
\end{aligned}
$$

Thus,

$$
\boxed{{}^G[I_W] = {}^W[I_W].}
$$

The frames still rotate relative to one another; it is the **wheel-inertia matrix** that stays numerically the same. This lets us describe both device inertias using the gimbal axes without tracking the wheel's transverse axes through every spin revolution.


**<ins>Inertias in the Spacecraft Body Frame</ins>**

To add momenta or multiply a matrix by a vector, we need their components in the same frame. We'll use body components unless a left superscript explicitly says otherwise.

The columns of the gimbal-to-body direction cosine matrix are the gimbal axes written in body components:

$$
[BG] =
\begin{bmatrix}
{}^B\hat{\mathbf g}_s & {}^B\hat{\mathbf g}_t & {}^B\hat{\mathbf g}_g
\end{bmatrix}.
$$

For any vector $\mathbf v$, this means ${}^B\mathbf v=[BG],{}^G\mathbf v$. Below, we omit the ${}^B$ superscript on the axis columns to keep the expressions readable.

Transform the gimbal inertia using the same matrix on both sides:

$$
\begin{aligned}
{}^B[I_G] &= [BG],{}^G[I_G],[BG]^T \\
&=
\begin{bmatrix}
I_{G_s}\hat{\mathbf g}_s & I_{G_t}\hat{\mathbf g}_t & I_{G_g}\hat{\mathbf g}_g
\end{bmatrix}
\begin{bmatrix}
\hat{\mathbf g}_s^T \\
\hat{\mathbf g}_t^T \\
\hat{\mathbf g}_g^T
\end{bmatrix} \\
&= I_{G_s}\hat{\mathbf g}_s\hat{\mathbf g}_s^T
    + I_{G_t}\hat{\mathbf g}_t\hat{\mathbf g}_t^T
    + I_{G_g}\hat{\mathbf g}_g\hat{\mathbf g}_g^T.
\end{aligned}
$$

Each outer product picks out one principal direction. For example, $\hat{\mathbf g}_s\hat{\mathbf g}_s^T\mathbf v$ projects $\mathbf v$ onto the spin axis and returns that projected vector.

The wheel transforms in exactly the same way, using the gimbal-frame matrix established above:

$$
\begin{aligned}
{}^B[I_W] &= [BG],{}^G[I_W],[BG]^T \\
&= I_{W_s}\hat{\mathbf g}_s\hat{\mathbf g}_s^T
    + I_{W_t}\hat{\mathbf g}_t\hat{\mathbf g}_t^T
    + I_{W_t}\hat{\mathbf g}_g\hat{\mathbf g}_g^T.
\end{aligned}
$$

From here on, $[I_G]$ and $[I_W]$ without component superscripts mean these body-frame matrices. Their principal moments are constant, but the outer products generally change as the gimbal turns.


**<ins>Body Rate Resolved Along the VSCMG Axes</ins>**

Let $\boldsymbol{\omega}\equiv\boldsymbol{\omega}_{B/N}$ be the spacecraft angular velocity relative to inertial space. We can resolve this same vector along either the body axes or the gimbal axes. For the device algebra, the gimbal directions are more convenient:

$$
\begin{aligned}
\omega_s &= \hat{\mathbf g}_s^T\boldsymbol{\omega}, \\
\omega_t &= \hat{\mathbf g}_t^T\boldsymbol{\omega}, \\
\omega_g &= \hat{\mathbf g}_g^T\boldsymbol{\omega}.
\end{aligned}
$$

These are scalar projections, so the reconstructed vector is

$$
\boxed{
\boldsymbol{\omega}
= \omega_s\hat{\mathbf g}_s + \omega_t\hat{\mathbf g}_t + \omega_g\hat{\mathbf g}_g.
}
$$

In coordinate form, the same relationship is

$$
{}^G\boldsymbol{\omega}=
\begin{bmatrix}
\omega_s \\ \omega_t \\ \omega_g
\end{bmatrix},
\qquad
{}^B\boldsymbol{\omega}=[BG],{}^G\boldsymbol{\omega}.
$$

Nothing about the physical angular velocity has changed. We have just chosen directions that make the device inertia multiplications easier to read. In particular, $\omega_s$ is the **body rate projected onto the spin axis**; it is distinct from the relative wheel spin rate $\Omega$.


**<ins>Spacecraft Angular Momentum</ins>**

The spacecraft body contribution is just the usual rigid-body result:

$$
\boxed{ \mathbf H_B = [I_s]\boldsymbol{\omega}. }
$$

Here, $[I_s]$ is the inertia of the spacecraft body excluding the gimbal and wheel inertias that are being modeled separately.

> **Reference consistency note:** the older angular-momentum section calls $[I_s]$ the inertia of the entire spacecraft including internal components, but its later EOM adds $[I_G]+[I_W]$ separately. Here, the gimbal and wheel rotational inertias are excluded from $[I_s]$, as required by that later derivation; fixed mass-offset contributions remain included once.


**<ins>Gimbal Angular Momentum</ins>**

Start with the gimbal's rigid-body momentum relation:

$$
\mathbf H_G=[I_G]\boldsymbol{\omega}_{G/N}.
$$

The angular velocity here must include the motion inherited from the spacecraft. Adding the two relative rotations gives

$$
\begin{aligned}
\boldsymbol{\omega}_{G/N}
&= \boldsymbol{\omega}_{G/B}+\boldsymbol{\omega}_{B/N} \\
&= \dot{\gamma}\hat{\mathbf g}_g+\boldsymbol{\omega} \\
&= \omega_s\hat{\mathbf g}_s+\omega_t\hat{\mathbf g}_t
    +(\omega_g+\dot{\gamma})\hat{\mathbf g}_g.
\end{aligned}
$$

All terms are vectors expressed in the same body components. Now substitute the dyadic inertia and distribute the multiplication:

$$
\begin{aligned}
\mathbf H_G
&= \left(I_{G_s}\hat{\mathbf g}_s\hat{\mathbf g}_s^T
    +I_{G_t}\hat{\mathbf g}_t\hat{\mathbf g}_t^T
    +I_{G_g}\hat{\mathbf g}_g\hat{\mathbf g}_g^T\right)
    (\boldsymbol{\omega}+\dot{\gamma}\hat{\mathbf g}_g) \\
&= [I_G]\boldsymbol{\omega}+[I_G](\dot{\gamma}\hat{\mathbf g}_g).
\end{aligned}
$$

**The body-rotation contribution.** Each outer product extracts the corresponding body-rate projection:

$$
\begin{aligned}
[I_G]\boldsymbol{\omega}
&= I_{G_s}\hat{\mathbf g}_s(\hat{\mathbf g}_s^T\boldsymbol{\omega})
    +I_{G_t}\hat{\mathbf g}_t(\hat{\mathbf g}_t^T\boldsymbol{\omega})
    +I_{G_g}\hat{\mathbf g}_g(\hat{\mathbf g}_g^T\boldsymbol{\omega}) \\
&= I_{G_s}\omega_s\hat{\mathbf g}_s
    +I_{G_t}\omega_t\hat{\mathbf g}_t
    +I_{G_g}\omega_g\hat{\mathbf g}_g.
\end{aligned}
$$

**The relative gimbal contribution.** Here orthogonality eliminates the spin and transverse terms:

$$
\begin{aligned}
[I_G](\dot{\gamma}\hat{\mathbf g}_g)
&= I_{G_s}\dot{\gamma}\hat{\mathbf g}_s
    \underbrace{(\hat{\mathbf g}_s^T\hat{\mathbf g}_g)}_{0} \\
&\quad +I_{G_t}\dot{\gamma}\hat{\mathbf g}_t
    \underbrace{(\hat{\mathbf g}_t^T\hat{\mathbf g}_g)}_{0} \\
&\quad +I_{G_g}\dot{\gamma}\hat{\mathbf g}_g
    \underbrace{(\hat{\mathbf g}_g^T\hat{\mathbf g}_g)}_{1} \\
&= I_{G_g}\dot{\gamma}\hat{\mathbf g}_g.
\end{aligned}
$$

Add the two contributions and collect the gimbal-axis rate:

$$
\boxed{
\mathbf H_G = I_{G_s}\omega_s\hat{\mathbf g}_s
    +I_{G_t}\omega_t\hat{\mathbf g}_t
    +I_{G_g}(\omega_g+\dot{\gamma})\hat{\mathbf g}_g.
}
$$

The gimbal carries momentum even when its motor is holding it still relative to the body. The $\omega_s$, $\omega_t$, and $\omega_g$ terms come from being carried around by the spacecraft; $\dot{\gamma}$ adds the relative motion about its own axis.


**<ins>Wheel Angular Momentum</ins>**

The wheel inherits the same spacecraft and gimbal motions, then adds its spin relative to $\mathcal G$:

$$
\begin{aligned}
\boldsymbol{\omega}_{W/N}
&= \boldsymbol{\omega}_{W/G}+\boldsymbol{\omega}_{G/B}+\boldsymbol{\omega}_{B/N} \\
&= \Omega\hat{\mathbf g}_s+\dot{\gamma}\hat{\mathbf g}_g+\boldsymbol{\omega} \\
&= (\omega_s+\Omega)\hat{\mathbf g}_s+\omega_t\hat{\mathbf g}_t
    +(\omega_g+\dot{\gamma})\hat{\mathbf g}_g.
\end{aligned}
$$

As before, multiply by the inertia of that part, and keep the three motions separate for a moment:

$$
\begin{aligned}
\mathbf H_W &= [I_W]\boldsymbol{\omega}_{W/N} \\
&= [I_W]\boldsymbol{\omega}
    +[I_W](\dot{\gamma}\hat{\mathbf g}_g)
    +[I_W](\Omega\hat{\mathbf g}_s).
\end{aligned}
$$

**1. Momentum carried by the body rotation.** The same dyadic multiplication used for the gimbal gives

$$
\begin{aligned}
[I_W]\boldsymbol{\omega}
&= \left(I_{W_s}\hat{\mathbf g}_s\hat{\mathbf g}_s^T
    +I_{W_t}\hat{\mathbf g}_t\hat{\mathbf g}_t^T
    +I_{W_t}\hat{\mathbf g}_g\hat{\mathbf g}_g^T\right)\boldsymbol{\omega} \\
&= I_{W_s}\omega_s\hat{\mathbf g}_s
    +I_{W_t}\omega_t\hat{\mathbf g}_t
    +I_{W_t}\omega_g\hat{\mathbf g}_g.
\end{aligned}
$$

**2. Momentum from relative gimbal motion.** In $\mathcal G$, this angular velocity has only a third component. Multiply there, then map the resulting column back to body components:

$$
\begin{aligned}
[I_W](\dot{\gamma}\hat{\mathbf g}_g)
&= [BG]
\begin{bmatrix}
I_{W_s} & 0 & 0 \\
0 & I_{W_t} & 0 \\
0 & 0 & I_{W_t}
\end{bmatrix}
\begin{bmatrix}
0 \\ 0 \\ \dot{\gamma}
\end{bmatrix} \\
&= [BG]
\begin{bmatrix}
0 \\ 0 \\ I_{W_t}\dot{\gamma}
\end{bmatrix} \\
&= I_{W_t}\dot{\gamma}\hat{\mathbf g}_g.
\end{aligned}
$$

**3. Momentum from relative wheel spin.** This time the angular velocity has only a first component. The wheel's axisymmetry lets us use its gimbal-frame inertia again:

$$
\begin{aligned}
[I_W](\Omega\hat{\mathbf g}_s)
&= [BG]
\begin{bmatrix}
I_{W_s} & 0 & 0 \\
0 & I_{W_t} & 0 \\
0 & 0 & I_{W_t}
\end{bmatrix}
\begin{bmatrix}
\Omega \\ 0 \\ 0
\end{bmatrix} \\
&= [BG]
\begin{bmatrix}
I_{W_s}\Omega \\ 0 \\ 0
\end{bmatrix} \\
&= I_{W_s}\Omega\hat{\mathbf g}_s.
\end{aligned}
$$

Putting the three pieces together,

$$
\begin{aligned}
\mathbf H_W
&= I_{W_s}\omega_s\hat{\mathbf g}_s
    +I_{W_t}\omega_t\hat{\mathbf g}_t
    +I_{W_t}\omega_g\hat{\mathbf g}_g \\
&\quad +I_{W_t}\dot{\gamma}\hat{\mathbf g}_g
    +I_{W_s}\Omega\hat{\mathbf g}_s.
\end{aligned}
$$

Therefore,

$$
\boxed{
\mathbf H_W = I_{W_s}(\omega_s+\Omega)\hat{\mathbf g}_s
    +I_{W_t}\omega_t\hat{\mathbf g}_t
    +I_{W_t}(\omega_g+\dot{\gamma})\hat{\mathbf g}_g.
}
$$

The relative spin contribution is along $\hat{\mathbf g}_s$, but the wheel's total momentum need not be. Its transverse inertia also responds to the spacecraft and gimbal motions. We keep those contributions throughout the derivation.


**<ins>Combined VSCMG Inertia</ins>**

The gimbal and wheel share the same principal directions in $\mathcal G$, so their inertias will keep appearing in pairs. Define

$$
[J]=[I_G]+[I_W].
$$

Add the matrices in the **same component frame**:

$$
\begin{aligned}
{}^G[J]
&= {}^G[I_G]+{}^G[I_W] \\
&=
\begin{bmatrix}
I_{G_s} & 0 & 0 \\
0 & I_{G_t} & 0 \\
0 & 0 & I_{G_g}
\end{bmatrix}
+\begin{bmatrix}
I_{W_s} & 0 & 0 \\
0 & I_{W_t} & 0 \\
0 & 0 & I_{W_t}
\end{bmatrix} \\
&=
\begin{bmatrix}
J_s & 0 & 0 \\
0 & J_t & 0 \\
0 & 0 & J_g
\end{bmatrix},
\end{aligned}
$$

where

$$
\boxed{
\begin{aligned}
J_s &= I_{G_s}+I_{W_s}, \\
J_t &= I_{G_t}+I_{W_t}, \\
J_g &= I_{G_g}+I_{W_t}.
\end{aligned}
}
$$

The combined moments are constant. The combined **body-frame matrix**, however, generally is not:

$$
\begin{aligned}
[J] &= [BG],{}^G[J],[BG]^T \\
&= J_s\hat{\mathbf g}_s\hat{\mathbf g}_s^T
    +J_t\hat{\mathbf g}_t\hat{\mathbf g}_t^T
    +J_g\hat{\mathbf g}_g\hat{\mathbf g}_g^T.
\end{aligned}
$$

In derivative notation,

$$
{}^G\frac{d[J]}{dt}=0,
\qquad
{}^B\frac{d[J]}{dt}\ne 0\quad\text{in general}.
$$

The difference is entirely in the moving axes. We will account for that motion when differentiating momentum, rather than treating $[J]$ as a body-fixed inertia.

> **Reference consistency note:** $\mathcal G$ and $\mathcal W$ generally do not coincide. We can add the inertias in $\mathcal G$ because axisymmetry gives ${}^G[I_W]={}^W[I_W]$. Also, the body-frame $[J]$ depends on gimbal angle $\gamma$, not on wheel spin rate $\Omega$.


**<ins>Total System Angular Momentum</ins>**

Now add the gimbal and wheel momenta, collecting one direction at a time. The spin-axis terms give

$$
\begin{aligned}
I_{G_s}\omega_s+I_{W_s}(\omega_s+\Omega)
&= (I_{G_s}+I_{W_s})\omega_s+I_{W_s}\Omega \\
&= J_s\omega_s+I_{W_s}\Omega.
\end{aligned}
$$

The transverse terms give

$$
I_{G_t}\omega_t+I_{W_t}\omega_t=J_t\omega_t,
$$

and the gimbal-axis terms give

$$
\begin{aligned}
I_{G_g}(\omega_g+\dot{\gamma})+I_{W_t}(\omega_g+\dot{\gamma})
&= (I_{G_g}+I_{W_t})(\omega_g+\dot{\gamma}) \\
&= J_g(\omega_g+\dot{\gamma}).
\end{aligned}
$$

So the combined device momentum is

$$
\begin{aligned}
\mathbf H_G+\mathbf H_W
&= (J_s\omega_s+I_{W_s}\Omega)\hat{\mathbf g}_s
    +J_t\omega_t\hat{\mathbf g}_t
    +J_g(\omega_g+\dot{\gamma})\hat{\mathbf g}_g \\
&= [J]\boldsymbol{\omega}
    +I_{W_s}\Omega\hat{\mathbf g}_s
    +J_g\dot{\gamma}\hat{\mathbf g}_g.
\end{aligned}
$$

Adding the spacecraft body gives the total:

$$
\boxed{
\mathbf H=([I_s]+[J])\boldsymbol{\omega}
    +I_{W_s}\Omega\hat{\mathbf g}_s
    +J_g\dot{\gamma}\hat{\mathbf g}_g.
}
$$

Written out in the same directions,

$$
\begin{aligned}
\mathbf H &= [I_s]\boldsymbol{\omega}
    +J_s\omega_s\hat{\mathbf g}_s+J_t\omega_t\hat{\mathbf g}_t \\
&\quad +J_g(\omega_g+\dot{\gamma})\hat{\mathbf g}_g
    +I_{W_s}\Omega\hat{\mathbf g}_s.
\end{aligned}
$$

That completes the momentum bookkeeping. Before differentiating, we need to finish the kinematics: how these axes move **inertially**, and how the body's scalar projections change as the axes turn.


**<ins>Inertial Rates of the Gimbal Axes</ins>**

Section 1.2 gave the axis rates seen from the spacecraft. An inertial observer also sees the spacecraft turning. For any vector $\mathbf v$, the transport theorem adds that extra motion:

$$
{}^N\frac{d\mathbf v}{dt}
= {}^B\frac{d\mathbf v}{dt}+\boldsymbol{\omega}\times\mathbf v.
$$

Here, the superscript on $d/dt$ identifies the **observing frame**. It does not require us to express the answer in that frame's coordinates; we continue using body components throughout the calculation.

The cyclic cross products are

$$
\hat{\mathbf g}_s\times\hat{\mathbf g}_t=\hat{\mathbf g}_g,
\qquad
\hat{\mathbf g}_t\times\hat{\mathbf g}_g=\hat{\mathbf g}_s,
\qquad
\hat{\mathbf g}_g\times\hat{\mathbf g}_s=\hat{\mathbf g}_t.
$$

Reversing a pair changes its sign. Let's use these explicitly for each axis.

**Spin direction.** Substitute its body-frame rate and expand the body angular velocity:

$$
\begin{aligned}
{}^N\frac{d\hat{\mathbf g}_s}{dt}
&= {}^B\frac{d\hat{\mathbf g}_s}{dt}
    +\boldsymbol{\omega}\times\hat{\mathbf g}_s \\
&= \dot{\gamma}\hat{\mathbf g}_t
    +(\omega_s\hat{\mathbf g}_s+\omega_t\hat{\mathbf g}_t
      +\omega_g\hat{\mathbf g}_g)\times\hat{\mathbf g}_s \\
&= \dot{\gamma}\hat{\mathbf g}_t
    -\omega_t\hat{\mathbf g}_g+\omega_g\hat{\mathbf g}_t \\
&= (\dot{\gamma}+\omega_g)\hat{\mathbf g}_t-\omega_t\hat{\mathbf g}_g.
\end{aligned}
$$

**Transverse direction.** The body-frame derivative has the opposite sign:

$$
\begin{aligned}
{}^N\frac{d\hat{\mathbf g}_t}{dt}
&= {}^B\frac{d\hat{\mathbf g}_t}{dt}
    +\boldsymbol{\omega}\times\hat{\mathbf g}_t \\
&= -\dot{\gamma}\hat{\mathbf g}_s
    +(\omega_s\hat{\mathbf g}_s+\omega_t\hat{\mathbf g}_t
      +\omega_g\hat{\mathbf g}_g)\times\hat{\mathbf g}_t \\
&= -\dot{\gamma}\hat{\mathbf g}_s
    +\omega_s\hat{\mathbf g}_g-\omega_g\hat{\mathbf g}_s \\
&= -(\dot{\gamma}+\omega_g)\hat{\mathbf g}_s+\omega_s\hat{\mathbf g}_g.
\end{aligned}
$$

**Gimbal direction.** Its body-frame derivative vanishes, but its inertial derivative generally does not:

$$
\begin{aligned}
{}^N\frac{d\hat{\mathbf g}_g}{dt}
&= \mathbf 0+\boldsymbol{\omega}\times\hat{\mathbf g}_g \\
&= (\omega_s\hat{\mathbf g}_s+\omega_t\hat{\mathbf g}_t
    +\omega_g\hat{\mathbf g}_g)\times\hat{\mathbf g}_g \\
&= -\omega_s\hat{\mathbf g}_t+\omega_t\hat{\mathbf g}_s.
\end{aligned}
$$

Keep these three results together for the momentum derivatives:

$$
\boxed{
\begin{aligned}
{}^N\frac{d\hat{\mathbf g}_s}{dt}
&= (\dot{\gamma}+\omega_g)\hat{\mathbf g}_t-\omega_t\hat{\mathbf g}_g, \\
{}^N\frac{d\hat{\mathbf g}_t}{dt}
&= -(\dot{\gamma}+\omega_g)\hat{\mathbf g}_s+\omega_s\hat{\mathbf g}_g, \\
{}^N\frac{d\hat{\mathbf g}_g}{dt}
&= \omega_t\hat{\mathbf g}_s-\omega_s\hat{\mathbf g}_t.
\end{aligned}
}
$$

Equivalently, each one is $\boldsymbol{\omega}_{G/N}\times\hat{\mathbf g}_i$. Notice that $\Omega$ does not appear: these are the gimbal axes, even when we use them to describe the spinning wheel's momentum.


**<ins>Rates of the Body-Rate Components</ins>**

There is one more moving-axis effect to keep track of. The scalar $\omega_s=\hat{\mathbf g}_s^T\boldsymbol{\omega}$ can change because the body angular velocity changes, because the spin axis moves, or both.

An overdot on a scalar means its ordinary time derivative. For the body angular velocity, we define

$$
\dot{\boldsymbol{\omega}}\equiv{}^B\frac{d\boldsymbol{\omega}}{dt}.
$$

Its inertial vector derivative happens to be the same:

$$
\begin{aligned}
{}^N\frac{d\boldsymbol{\omega}}{dt}
&= {}^B\frac{d\boldsymbol{\omega}}{dt}
    +\boldsymbol{\omega}\times\boldsymbol{\omega} \\
&= \dot{\boldsymbol{\omega}}.
\end{aligned}
$$

That equality is special to $\boldsymbol{\omega}_{B/N}$; we cannot use it for a general vector or for the moving axes.

**Spin-axis component.** Apply the product rule using body-frame derivatives for both vectors:

$$
\begin{aligned}
\dot{\omega}_s
&= \frac{d}{dt}(\hat{\mathbf g}_s^T\boldsymbol{\omega}) \\
&= \left({}^B\frac{d\hat{\mathbf g}_s}{dt}\right)^T\boldsymbol{\omega}
    +\hat{\mathbf g}_s^T\dot{\boldsymbol{\omega}} \\
&= \dot{\gamma}\hat{\mathbf g}_t^T\boldsymbol{\omega}
    +\hat{\mathbf g}_s^T\dot{\boldsymbol{\omega}} \\
&= \dot{\gamma}\omega_t+\hat{\mathbf g}_s^T\dot{\boldsymbol{\omega}}.
\end{aligned}
$$

**Transverse component.** Use ${}^B d\hat{\mathbf g}_t/dt=-\dot{\gamma}\hat{\mathbf g}_s$:

$$
\begin{aligned}
\dot{\omega}_t
&= \left({}^B\frac{d\hat{\mathbf g}_t}{dt}\right)^T\boldsymbol{\omega}
    +\hat{\mathbf g}_t^T\dot{\boldsymbol{\omega}} \\
&= -\dot{\gamma}\hat{\mathbf g}_s^T\boldsymbol{\omega}
    +\hat{\mathbf g}_t^T\dot{\boldsymbol{\omega}} \\
&= -\dot{\gamma}\omega_s+\hat{\mathbf g}_t^T\dot{\boldsymbol{\omega}}.
\end{aligned}
$$

**Gimbal component.** This direction stays fixed in the body:

$$
\begin{aligned}
\dot{\omega}_g
&= \left({}^B\frac{d\hat{\mathbf g}_g}{dt}\right)^T\boldsymbol{\omega}
    +\hat{\mathbf g}_g^T\dot{\boldsymbol{\omega}} \\
&= \hat{\mathbf g}_g^T\dot{\boldsymbol{\omega}}.
\end{aligned}
$$

To keep the later component equations readable, give the **angular-acceleration projections** short names:

$$
\alpha_s\equiv\hat{\mathbf g}_s^T\dot{\boldsymbol{\omega}},
\qquad
\alpha_t\equiv\hat{\mathbf g}_t^T\dot{\boldsymbol{\omega}},
\qquad
\alpha_g\equiv\hat{\mathbf g}_g^T\dot{\boldsymbol{\omega}}.
$$

Then the three identities are

$$
\boxed{
\begin{aligned}
\dot{\omega}_s &= \alpha_s+\dot{\gamma}\omega_t, \\
\dot{\omega}_t &= \alpha_t-\dot{\gamma}\omega_s, \\
\dot{\omega}_g &= \alpha_g.
\end{aligned}
}
$$

The distinction matters: $\alpha_s$ is a projection of angular acceleration, while $\dot{\omega}_s$ is the derivative of a projection onto a moving axis. Even if $\dot{\boldsymbol{\omega}}=\mathbf 0$, gimbal motion can change $\omega_s$ and $\omega_t$.


**<ins>Quick Frame-Derivative Check</ins>**

We should get the same scalar rates if we differentiate inertially. Take the transverse projection as a check:

$$
\begin{aligned}
\dot{\omega}_t
&= \left({}^N\frac{d\hat{\mathbf g}_t}{dt}\right)^T\boldsymbol{\omega}
    +\hat{\mathbf g}_t^T\,{}^N\frac{d\boldsymbol{\omega}}{dt} \\
&= \left[-(\dot{\gamma}+\omega_g)\hat{\mathbf g}_s
    +\omega_s\hat{\mathbf g}_g\right]^T\boldsymbol{\omega}+\alpha_t \\
&= -(\dot{\gamma}+\omega_g)\omega_s+\omega_s\omega_g+\alpha_t \\
&= -\dot{\gamma}\omega_s+\alpha_t.
\end{aligned}
$$

The $\omega_s\omega_g$ terms cancel. More generally, the extra transport contribution to a scalar projection is

$$
(\boldsymbol{\omega}\times\hat{\mathbf g}_i)^T\boldsymbol{\omega}=0,
$$

because the cross product is perpendicular to $\boldsymbol{\omega}$. So either observing frame gives the same scalar derivative, provided both vectors in the product rule use that frame consistently.

We now have everything needed for the EOM: the three momenta, the inertial basis rates, and the scalar component rates. The next step is to put those pieces through the product rule.


## 1.4 - Single-VSCMG EOM

**<ins>How We Will Organize the Momentum Derivatives</ins>**

The spacecraft equation is the inertial momentum balance

$$
{}^N\frac{d\mathbf H_B}{dt}
+{}^N\frac{d\mathbf H_G}{dt}
+{}^N\frac{d\mathbf H_W}{dt}=\mathbf L.
$$

We'll differentiate each part separately, then collect their common terms. For the gimbal and wheel, it helps to keep the **vector derivative** separate from its **directional components**. Define

$$
\mathbf D_G\equiv{}^N\frac{d\mathbf H_G}{dt},
\qquad
\mathbf D_W\equiv{}^N\frac{d\mathbf H_W}{dt}.
$$

For either part $X\in\{G,W\}$, write

$$
D_{X,s}=\hat{\mathbf g}_s^T\mathbf D_X,
\qquad
D_{X,t}=\hat{\mathbf g}_t^T\mathbf D_X,
\qquad
D_{X,g}=\hat{\mathbf g}_g^T\mathbf D_X.
$$

The first index identifies the **part**; the second identifies the **direction**. For example, $D_{G,s}$ is the gimbal's inertial momentum rate projected onto the spin axis. All these components have units of torque.

After working through the three directions, the vector is simply

$$
\mathbf D_X=D_{X,s}\hat{\mathbf g}_s
    +D_{X,t}\hat{\mathbf g}_t+D_{X,g}\hat{\mathbf g}_g.
$$

One detail to keep straight: projecting the derivative is different from differentiating a moving-axis projection. If $H_{X,s}=\hat{\mathbf g}_s^T\mathbf H_X$, then

$$
\frac{dH_{X,s}}{dt}
=\left({}^N\frac{d\hat{\mathbf g}_s}{dt}\right)^T\mathbf H_X+D_{X,s}.
$$

Using $D_{X,s}$ avoids hiding that extra basis-motion term in an ambiguous dotted component.


**<ins>Inertial Derivative of Spacecraft Angular Momentum</ins>**

The spacecraft contribution is the familiar rigid-body one. Start with $\mathbf H_B=[I_s]\boldsymbol{\omega}$ and apply the body-to-inertial transport theorem:

$$
\begin{aligned}
{}^N\frac{d\mathbf H_B}{dt}
&= {}^B\frac{d([I_s]\boldsymbol{\omega})}{dt}
    +\boldsymbol{\omega}\times([I_s]\boldsymbol{\omega}) \\
&= \left({}^B\frac{d[I_s]}{dt}\right)\boldsymbol{\omega}
    +[I_s]\dot{\boldsymbol{\omega}}
    +\boldsymbol{\omega}\times([I_s]\boldsymbol{\omega}).
\end{aligned}
$$

Since $[I_s]$ is constant in body components, its body-frame derivative vanishes:

$$
\boxed{
{}^N\frac{d\mathbf H_B}{dt}
=[I_s]\dot{\boldsymbol{\omega}}
    +\boldsymbol{\omega}\times([I_s]\boldsymbol{\omega}).
}
$$

The first term comes from changing body rate; the cross product comes from carrying the momentum around with the rotating body basis. For the gimbal and wheel, we'll account for that basis motion directly using the inertial axis derivatives already worked out in Section 1.3.


**<ins>Inertial Derivative of Gimbal Angular Momentum</ins>**

Start from the gimbal momentum derived in Section 1.3:

$$
\mathbf H_G=I_{G_s}\omega_s\hat{\mathbf g}_s
    +I_{G_t}\omega_t\hat{\mathbf g}_t
    +I_{G_g}(\omega_g+\dot{\gamma})\hat{\mathbf g}_g.
$$

The principal moments are constant, but each scalar rate and each direction can change. Apply the product rule to all three terms. Here $\ddot{\gamma}=d\dot{\gamma}/dt$ is the relative gimbal acceleration.

$$
\begin{aligned}
\mathbf D_G
&= I_{G_s}\left(\dot{\omega}_s\hat{\mathbf g}_s
    +\omega_s\,{}^N\frac{d\hat{\mathbf g}_s}{dt}\right) \\
&\quad +I_{G_t}\left(\dot{\omega}_t\hat{\mathbf g}_t
    +\omega_t\,{}^N\frac{d\hat{\mathbf g}_t}{dt}\right) \\
&\quad +I_{G_g}\left[(\dot{\omega}_g+\ddot{\gamma})\hat{\mathbf g}_g
    +(\omega_g+\dot{\gamma})\,{}^N\frac{d\hat{\mathbf g}_g}{dt}\right].
\end{aligned}
$$

Now substitute the inertial axis rates. Keep each original momentum term together so we can see where its contributions go:

$$
\begin{aligned}
\mathbf D_G
&= I_{G_s}\dot{\omega}_s\hat{\mathbf g}_s
    +I_{G_s}\omega_s\left[(\dot{\gamma}+\omega_g)\hat{\mathbf g}_t
      -\omega_t\hat{\mathbf g}_g\right] \\
&\quad +I_{G_t}\dot{\omega}_t\hat{\mathbf g}_t
    +I_{G_t}\omega_t\left[-(\dot{\gamma}+\omega_g)\hat{\mathbf g}_s
      +\omega_s\hat{\mathbf g}_g\right] \\
&\quad +I_{G_g}(\dot{\omega}_g+\ddot{\gamma})\hat{\mathbf g}_g
    +I_{G_g}(\omega_g+\dot{\gamma})
      (\omega_t\hat{\mathbf g}_s-\omega_s\hat{\mathbf g}_t).
\end{aligned}
$$

At this point the vector differentiation is done. The remaining work is to gather the coefficients of each direction and substitute the scalar-rate identities.

**Spin direction — $D_{G,s}$.** Read off every term multiplying $\hat{\mathbf g}_s$:

$$
\begin{aligned}
D_{G,s}
&= I_{G_s}\dot{\omega}_s
    -I_{G_t}\omega_t(\dot{\gamma}+\omega_g)
    +I_{G_g}(\omega_g+\dot{\gamma})\omega_t \\
&= I_{G_s}\dot{\omega}_s
    +(I_{G_g}-I_{G_t})(\dot{\gamma}+\omega_g)\omega_t.
\end{aligned}
$$

Use $\dot{\omega}_s=\alpha_s+\dot{\gamma}\omega_t$, then collect the gimbal-rate and body-rate products separately:

$$
\begin{aligned}
D_{G,s}
&= I_{G_s}(\alpha_s+\dot{\gamma}\omega_t)
    +(I_{G_g}-I_{G_t})(\dot{\gamma}\omega_t+\omega_t\omega_g) \\
&= I_{G_s}\alpha_s
    +(I_{G_s}-I_{G_t}+I_{G_g})\dot{\gamma}\omega_t
    +(I_{G_g}-I_{G_t})\omega_t\omega_g.
\end{aligned}
$$

**Transverse direction — $D_{G,t}$.** The same collection gives

$$
\begin{aligned}
D_{G,t}
&= I_{G_t}\dot{\omega}_t
    +I_{G_s}\omega_s(\dot{\gamma}+\omega_g)
    -I_{G_g}(\omega_g+\dot{\gamma})\omega_s \\
&= I_{G_t}\dot{\omega}_t
    +(I_{G_s}-I_{G_g})(\dot{\gamma}+\omega_g)\omega_s.
\end{aligned}
$$

This time $\dot{\omega}_t=\alpha_t-\dot{\gamma}\omega_s$ carries a minus sign:

$$
\begin{aligned}
D_{G,t}
&= I_{G_t}(\alpha_t-\dot{\gamma}\omega_s)
    +(I_{G_s}-I_{G_g})(\dot{\gamma}\omega_s+\omega_s\omega_g) \\
&= I_{G_t}\alpha_t
    +(I_{G_s}-I_{G_t}-I_{G_g})\dot{\gamma}\omega_s
    +(I_{G_s}-I_{G_g})\omega_s\omega_g.
\end{aligned}
$$

**Gimbal direction — $D_{G,g}$.** The spin and transverse momentum directions both contribute here:

$$
\begin{aligned}
D_{G,g}
&= -I_{G_s}\omega_s\omega_t+I_{G_t}\omega_t\omega_s
    +I_{G_g}(\dot{\omega}_g+\ddot{\gamma}) \\
&= I_{G_g}(\alpha_g+\ddot{\gamma})
    +(I_{G_t}-I_{G_s})\omega_s\omega_t.
\end{aligned}
$$

**Put the directions back together.** The three rows below are ordered spin, transverse, gimbal. Multiplying by $[BG]$ reconstructs the body-component vector:

$$
\boxed{
\mathbf D_G=[BG]
\begin{bmatrix}
I_{G_s}\alpha_s+(I_{G_s}-I_{G_t}+I_{G_g})\dot{\gamma}\omega_t
    +(I_{G_g}-I_{G_t})\omega_t\omega_g \\
I_{G_t}\alpha_t+(I_{G_s}-I_{G_t}-I_{G_g})\dot{\gamma}\omega_s
    +(I_{G_s}-I_{G_g})\omega_s\omega_g \\
I_{G_g}(\alpha_g+\ddot{\gamma})+(I_{G_t}-I_{G_s})\omega_s\omega_t
\end{bmatrix}.
}
$$

This is the same vector as $D_{G,s}\hat{\mathbf g}_s+D_{G,t}\hat{\mathbf g}_t+D_{G,g}\hat{\mathbf g}_g$. Its terms fall into three useful groups: spacecraft angular acceleration, products of body rates, and relative gimbal motion. That grouping will make the final addition much easier.


**<ins>Inertial Derivative of Wheel Angular Momentum</ins>**

The wheel follows the same steps, with the extra relative spin carried in its first component:

$$
\mathbf H_W=I_{W_s}(\omega_s+\Omega)\hat{\mathbf g}_s
    +I_{W_t}\omega_t\hat{\mathbf g}_t
    +I_{W_t}(\omega_g+\dot{\gamma})\hat{\mathbf g}_g.
$$

Let $\dot{\Omega}=d\Omega/dt$ be the relative wheel spin acceleration. Differentiating both the coefficients and the axes gives

$$
\begin{aligned}
\mathbf D_W
&= I_{W_s}(\dot{\omega}_s+\dot{\Omega})\hat{\mathbf g}_s
    +I_{W_s}(\omega_s+\Omega)\,{}^N\frac{d\hat{\mathbf g}_s}{dt} \\
&\quad +I_{W_t}\dot{\omega}_t\hat{\mathbf g}_t
    +I_{W_t}\omega_t\,{}^N\frac{d\hat{\mathbf g}_t}{dt} \\
&\quad +I_{W_t}(\dot{\omega}_g+\ddot{\gamma})\hat{\mathbf g}_g
    +I_{W_t}(\omega_g+\dot{\gamma})\,{}^N\frac{d\hat{\mathbf g}_g}{dt}.
\end{aligned}
$$

Substitute the gimbal-basis derivatives from Section 1.3:

$$
\begin{aligned}
\mathbf D_W
&= I_{W_s}(\dot{\omega}_s+\dot{\Omega})\hat{\mathbf g}_s
    +I_{W_s}(\omega_s+\Omega)
      \left[(\dot{\gamma}+\omega_g)\hat{\mathbf g}_t-\omega_t\hat{\mathbf g}_g\right] \\
&\quad +I_{W_t}\dot{\omega}_t\hat{\mathbf g}_t
    +I_{W_t}\omega_t
      \left[-(\dot{\gamma}+\omega_g)\hat{\mathbf g}_s+\omega_s\hat{\mathbf g}_g\right] \\
&\quad +I_{W_t}(\dot{\omega}_g+\ddot{\gamma})\hat{\mathbf g}_g
    +I_{W_t}(\omega_g+\dot{\gamma})
      (\omega_t\hat{\mathbf g}_s-\omega_s\hat{\mathbf g}_t).
\end{aligned}
$$

These are already inertial derivatives. We do not add another $\boldsymbol{\omega}\times\mathbf H_W$ afterward; that transport is included in the axis rates.

**Spin direction — $D_{W,s}$.** Gather the three contributions along $\hat{\mathbf g}_s$:

$$
\begin{aligned}
D_{W,s}
&= I_{W_s}(\dot{\omega}_s+\dot{\Omega})
    -I_{W_t}\omega_t(\dot{\gamma}+\omega_g)
    +I_{W_t}(\omega_g+\dot{\gamma})\omega_t \\
&= I_{W_s}(\dot{\omega}_s+\dot{\Omega}) \\
&= I_{W_s}(\alpha_s+\dot{\gamma}\omega_t+\dot{\Omega}).
\end{aligned}
$$

The two transverse-inertia terms cancel exactly because the wheel's transverse moments are equal. No small-term approximation is involved.

**Transverse direction — $D_{W,t}$.** Before substituting scalar derivatives,

$$
D_{W,t}=I_{W_t}\dot{\omega}_t
    +I_{W_s}(\omega_s+\Omega)(\dot{\gamma}+\omega_g)
    -I_{W_t}(\omega_g+\dot{\gamma})\omega_s.
$$

Use $\dot{\omega}_t=\alpha_t-\dot{\gamma}\omega_s$ and expand the spin product. The longer step is split into the contributions that will be collected together:

$$
\begin{aligned}
D_{W,t}
&= I_{W_t}(\alpha_t-\dot{\gamma}\omega_s)
    +I_{W_s}(\omega_s+\Omega)(\dot{\gamma}+\omega_g)
    -I_{W_t}(\omega_g+\dot{\gamma})\omega_s \\
&= I_{W_t}\alpha_t-I_{W_t}\dot{\gamma}\omega_s \\
&\quad +I_{W_s}\dot{\gamma}\omega_s+I_{W_s}\Omega\dot{\gamma}
    +I_{W_s}\omega_s\omega_g+I_{W_s}\Omega\omega_g \\
&\quad -I_{W_t}\omega_s\omega_g-I_{W_t}\dot{\gamma}\omega_s \\
&= I_{W_t}\alpha_t+(I_{W_s}-2I_{W_t})\dot{\gamma}\omega_s \\
&\quad +(I_{W_s}-I_{W_t})\omega_s\omega_g
    +I_{W_s}\Omega(\dot{\gamma}+\omega_g).
\end{aligned}
$$

The factor $2I_{W_t}$ comes from two places: the moving-axis correction in $\dot{\omega}_t$, and the turning of the wheel's gimbal-axis momentum. It's easy to lose one if we skip the expansion.

**Gimbal direction — $D_{W,g}$.** Collect, substitute $\dot{\omega}_g=\alpha_g$, and expand the spin term:

$$
\begin{aligned}
D_{W,g}
&= I_{W_t}(\dot{\omega}_g+\ddot{\gamma})
    -I_{W_s}(\omega_s+\Omega)\omega_t+I_{W_t}\omega_t\omega_s \\
&= I_{W_t}(\alpha_g+\ddot{\gamma})
    -I_{W_s}\omega_s\omega_t-I_{W_s}\Omega\omega_t
    +I_{W_t}\omega_s\omega_t \\
&= I_{W_t}(\alpha_g+\ddot{\gamma})
    +(I_{W_t}-I_{W_s})\omega_s\omega_t-I_{W_s}\Omega\omega_t.
\end{aligned}
$$

**Put the directions back together.** Again, the rows are spin, transverse, gimbal:

$$
\boxed{
\mathbf D_W=[BG]
\begin{bmatrix}
I_{W_s}(\alpha_s+\dot{\gamma}\omega_t+\dot{\Omega}) \\
I_{W_t}\alpha_t+(I_{W_s}-2I_{W_t})\dot{\gamma}\omega_s
    +(I_{W_s}-I_{W_t})\omega_s\omega_g
    +I_{W_s}\Omega(\dot{\gamma}+\omega_g) \\
I_{W_t}(\alpha_g+\ddot{\gamma})
    +(I_{W_t}-I_{W_s})\omega_s\omega_t-I_{W_s}\Omega\omega_t
\end{bmatrix}.
}
$$

The familiar actuation terms are now visible: $I_{W_s}\dot{\Omega}$ changes spin momentum, and $I_{W_s}\Omega\dot{\gamma}$ turns it toward the transverse direction. The $\Omega\omega_g$ and $\Omega\omega_t$ terms also remain, since spacecraft rotation can turn that momentum even when the gimbal is held still.


**<ins>Collecting the Spacecraft Angular-Acceleration Terms</ins>**

Now combine the spacecraft derivative with $\mathbf D_G$ and $\mathbf D_W$:

$$
{}^N\frac{d\mathbf H_B}{dt}+\mathbf D_G+\mathbf D_W=\mathbf L.
$$

Internal motor and bearing torques cancel in this whole-system balance. Rather than add every term at once, we'll collect three groups: angular-acceleration terms, products of body rates, and the terms involving relative device motion.

Start with the coefficients of $\alpha_s$, $\alpha_t$, and $\alpha_g$ in the gimbal and wheel results:

$$
\begin{aligned}
&(I_{G_s}+I_{W_s})\alpha_s\hat{\mathbf g}_s
    +(I_{G_t}+I_{W_t})\alpha_t\hat{\mathbf g}_t
    +(I_{G_g}+I_{W_t})\alpha_g\hat{\mathbf g}_g \\
&\qquad = J_s\alpha_s\hat{\mathbf g}_s
    +J_t\alpha_t\hat{\mathbf g}_t+J_g\alpha_g\hat{\mathbf g}_g.
\end{aligned}
$$

Restore the definitions $\alpha_i=\hat{\mathbf g}_i^T\dot{\boldsymbol{\omega}}$ to turn these directional terms back into a matrix multiplying spacecraft acceleration:

$$
\begin{aligned}
J_s\alpha_s\hat{\mathbf g}_s+J_t\alpha_t\hat{\mathbf g}_t+J_g\alpha_g\hat{\mathbf g}_g
&= \left(J_s\hat{\mathbf g}_s\hat{\mathbf g}_s^T
    +J_t\hat{\mathbf g}_t\hat{\mathbf g}_t^T
    +J_g\hat{\mathbf g}_g\hat{\mathbf g}_g^T\right)\dot{\boldsymbol{\omega}} \\
&= [J]\dot{\boldsymbol{\omega}}.
\end{aligned}
$$

Adding the spacecraft term gives

$$
[I_s]\dot{\boldsymbol{\omega}}+[J]\dot{\boldsymbol{\omega}}
=([I_s]+[J])\dot{\boldsymbol{\omega}}.
$$

Define the total instantaneous inertia in body components:

$$
\boxed{[I]=[I_s]+[J].}
$$

All the spacecraft angular-acceleration terms are therefore $[I]\dot{\boldsymbol{\omega}}$. This grouping does not make $[I]$ constant: the moving-inertia effects are still present in the other terms we have yet to collect.


**<ins>Collecting the Products of Body Rates</ins>**

Next take the terms containing two body-rate components, with no $\Omega$ or relative gimbal rate. The directional results make it easy to track the contribution from each part.

**Spin direction.** The gimbal contributes $(I_{G_g}-I_{G_t})\omega_t\omega_g$. The wheel contributes zero because its transverse moments cancel:

$$
\begin{aligned}
(I_{G_g}-I_{G_t})\omega_t\omega_g
&= [(I_{G_g}+I_{W_t})-(I_{G_t}+I_{W_t})]\omega_t\omega_g \\
&= (J_g-J_t)\omega_t\omega_g.
\end{aligned}
$$

**Transverse direction.** Add the gimbal and wheel coefficients:

$$
\begin{aligned}
[(I_{G_s}-I_{G_g})+(I_{W_s}-I_{W_t})]\omega_s\omega_g
&= [(I_{G_s}+I_{W_s})-(I_{G_g}+I_{W_t})]\omega_s\omega_g \\
&= (J_s-J_g)\omega_s\omega_g.
\end{aligned}
$$

**Gimbal direction.** The same grouping gives

$$
\begin{aligned}
[(I_{G_t}-I_{G_s})+(I_{W_t}-I_{W_s})]\omega_s\omega_t
&= [(I_{G_t}+I_{W_t})-(I_{G_s}+I_{W_s})]\omega_s\omega_t \\
&= (J_t-J_s)\omega_s\omega_t.
\end{aligned}
$$

These coefficients have the shape of an Euler gyroscopic term. Let's check that by expanding $\boldsymbol{\omega}\times([J]\boldsymbol{\omega})$ directly. First,

$$
[J]\boldsymbol{\omega}=J_s\omega_s\hat{\mathbf g}_s
    +J_t\omega_t\hat{\mathbf g}_t+J_g\omega_g\hat{\mathbf g}_g.
$$

Cross this with $\boldsymbol{\omega}=\omega_s\hat{\mathbf g}_s+\omega_t\hat{\mathbf g}_t+\omega_g\hat{\mathbf g}_g$. The three same-axis products vanish; the six remaining products come in opposite-sign pairs:

$$
\begin{aligned}
\boldsymbol{\omega}\times([J]\boldsymbol{\omega})
&= J_g\omega_t\omega_g(\hat{\mathbf g}_t\times\hat{\mathbf g}_g)
    +J_t\omega_g\omega_t(\hat{\mathbf g}_g\times\hat{\mathbf g}_t) \\
&\quad +J_s\omega_g\omega_s(\hat{\mathbf g}_g\times\hat{\mathbf g}_s)
    +J_g\omega_s\omega_g(\hat{\mathbf g}_s\times\hat{\mathbf g}_g) \\
&\quad +J_t\omega_s\omega_t(\hat{\mathbf g}_s\times\hat{\mathbf g}_t)
    +J_s\omega_t\omega_s(\hat{\mathbf g}_t\times\hat{\mathbf g}_s) \\
&= (J_g-J_t)\omega_t\omega_g\hat{\mathbf g}_s \\
&\quad +(J_s-J_g)\omega_s\omega_g\hat{\mathbf g}_t \\
&\quad +(J_t-J_s)\omega_s\omega_t\hat{\mathbf g}_g.
\end{aligned}
$$

That matches the three collected coefficients, including their signs. With the spacecraft's own gyroscopic term,

$$
\begin{aligned}
\boldsymbol{\omega}\times([I_s]\boldsymbol{\omega})
+\boldsymbol{\omega}\times([J]\boldsymbol{\omega})
&= \boldsymbol{\omega}\times\left(([I_s]+[J])\boldsymbol{\omega}\right) \\
&= \boldsymbol{\omega}\times([I]\boldsymbol{\omega}).
\end{aligned}
$$

So this whole group becomes the familiar gyroscopic cross product, evaluated using the instantaneous total inertia.


**<ins>Remaining Device Terms and the Single-VSCMG EOM</ins>**

The terms left over contain relative gimbal motion or wheel spin. Call their coefficients $c_s$, $c_t$, and $c_g$, according to direction. These are contributions to the total momentum rate, with units of torque; we will calculate the actual motor torques separately in Section 1.5.

**Spin direction.** Add the remaining terms from $D_{G,s}$ and $D_{W,s}$:

$$
\begin{aligned}
c_s
&= (I_{G_s}-I_{G_t}+I_{G_g})\dot{\gamma}\omega_t
    +I_{W_s}\dot{\gamma}\omega_t+I_{W_s}\dot{\Omega} \\
&= [(I_{G_s}+I_{W_s})-(I_{G_t}+I_{W_t})+(I_{G_g}+I_{W_t})]
    \dot{\gamma}\omega_t+I_{W_s}\dot{\Omega} \\
&= (J_s-J_t+J_g)\dot{\gamma}\omega_t+I_{W_s}\dot{\Omega}.
\end{aligned}
$$

**Transverse direction.** The two gimbal-rate coefficients combine, while the spin-momentum terms stay explicit:

$$
\begin{aligned}
c_t
&= (I_{G_s}-I_{G_t}-I_{G_g})\dot{\gamma}\omega_s
    +(I_{W_s}-2I_{W_t})\dot{\gamma}\omega_s
    +I_{W_s}\Omega(\dot{\gamma}+\omega_g) \\
&= [(I_{G_s}+I_{W_s})-(I_{G_t}+I_{W_t})-(I_{G_g}+I_{W_t})]
    \dot{\gamma}\omega_s+I_{W_s}\Omega(\dot{\gamma}+\omega_g) \\
&= (J_s-J_t-J_g)\dot{\gamma}\omega_s+I_{W_s}\Omega(\dot{\gamma}+\omega_g).
\end{aligned}
$$

**Gimbal direction.** Here the relative acceleration and spin coupling remain:

$$
\begin{aligned}
c_g
&= (I_{G_g}+I_{W_t})\ddot{\gamma}-I_{W_s}\Omega\omega_t \\
&= J_g\ddot{\gamma}-I_{W_s}\Omega\omega_t.
\end{aligned}
$$

We can now write the complete inertial momentum rate using the three groups just derived:

$$
\begin{aligned}
{}^N\frac{d\mathbf H}{dt}
&= [I]\dot{\boldsymbol{\omega}}
    +\boldsymbol{\omega}\times([I]\boldsymbol{\omega}) \\
&\quad +c_s\hat{\mathbf g}_s+c_t\hat{\mathbf g}_t+c_g\hat{\mathbf g}_g \\
&= \mathbf L.
\end{aligned}
$$

Move the gyroscopic and device terms to the right:

$$
\begin{aligned}
[I]\dot{\boldsymbol{\omega}}
&= \mathbf L-\boldsymbol{\omega}\times([I]\boldsymbol{\omega}) \\
&\quad -c_s\hat{\mathbf g}_s-c_t\hat{\mathbf g}_t-c_g\hat{\mathbf g}_g.
\end{aligned}
$$

Finally, substitute the coefficients. This is the full **single-VSCMG spacecraft EOM**:

$$
\boxed{
\begin{aligned}
[I]\dot{\boldsymbol{\omega}}
&= \mathbf L-\boldsymbol{\omega}\times([I]\boldsymbol{\omega}) \\
&\quad -\hat{\mathbf g}_s
    \left[(J_s-J_t+J_g)\dot{\gamma}\omega_t+I_{W_s}\dot{\Omega}\right] \\
&\quad -\hat{\mathbf g}_t
    \left[(J_s-J_t-J_g)\dot{\gamma}\omega_s
      +I_{W_s}\Omega(\dot{\gamma}+\omega_g)\right] \\
&\quad -\hat{\mathbf g}_g
    \left[J_g\ddot{\gamma}-I_{W_s}\Omega\omega_t\right].
\end{aligned}
}
$$

Every term has a traceable source: the inertia multiplication, a changing scalar rate, or a moving momentum direction. We have kept the transverse inertias and all body-rate couplings; there is no large-spin or small-body-rate approximation here.

For comparison with the reference's grouping, the same spin and transverse coefficients can be written as

$$
\begin{aligned}
c_s &= J_s\dot{\gamma}\omega_t+I_{W_s}\dot{\Omega}
    -(J_t-J_g)\omega_t\dot{\gamma}, \\
c_t &= (J_s\omega_s+I_{W_s}\Omega)\dot{\gamma}
    -(J_t+J_g)\omega_s\dot{\gamma}+I_{W_s}\Omega\omega_g.
\end{aligned}
$$

If the device motions are prescribed, this equation gives the spacecraft acceleration. If the inputs are motor torques, the internal accelerations $\dot{\Omega}$ and $\ddot{\gamma}$ must be solved together with $\dot{\boldsymbol{\omega}}$. That's the purpose of the next section.


**<ins>Checking the Changing Inertia and Limiting Cases</ins>**

Before moving on, let's check the moving-inertia terms through the compact momentum expression. This is a second route to the same result:

$$
\mathbf H=[I]\boldsymbol{\omega}
    +J_g\dot{\gamma}\hat{\mathbf g}_g
    +I_{W_s}\Omega\hat{\mathbf g}_s.
$$

Since $[I_s]$ is body-fixed, ${}^B d[I]/dt={}^B d[J]/dt$. Differentiate the dyadic form of $[J]$ using the product rule and the body-frame basis rates:

$$
\begin{aligned}
{}^B\frac{d[I]}{dt}
&= J_s\left[
    \left({}^B\frac{d\hat{\mathbf g}_s}{dt}\right)\hat{\mathbf g}_s^T
    +\hat{\mathbf g}_s\left({}^B\frac{d\hat{\mathbf g}_s}{dt}\right)^T\right] \\
&\quad +J_t\left[
    \left({}^B\frac{d\hat{\mathbf g}_t}{dt}\right)\hat{\mathbf g}_t^T
    +\hat{\mathbf g}_t\left({}^B\frac{d\hat{\mathbf g}_t}{dt}\right)^T\right] \\
&= J_s\dot{\gamma}(\hat{\mathbf g}_t\hat{\mathbf g}_s^T
    +\hat{\mathbf g}_s\hat{\mathbf g}_t^T) \\
&\quad -J_t\dot{\gamma}(\hat{\mathbf g}_s\hat{\mathbf g}_t^T
    +\hat{\mathbf g}_t\hat{\mathbf g}_s^T) \\
&= (J_s-J_t)\dot{\gamma}(\hat{\mathbf g}_t\hat{\mathbf g}_s^T
    +\hat{\mathbf g}_s\hat{\mathbf g}_t^T).
\end{aligned}
$$

The $J_g$ dyad has zero body-frame derivative because $\hat{\mathbf g}_g$ is body-fixed. Multiplying the remaining expression by $\boldsymbol{\omega}$ gives

$$
\left({}^B\frac{d[I]}{dt}\right)\boldsymbol{\omega}
=(J_s-J_t)\dot{\gamma}
    (\omega_s\hat{\mathbf g}_t+\omega_t\hat{\mathbf g}_s).
$$

Now differentiate the compact momentum in the body frame, then add the transport term:

$$
\begin{aligned}
{}^N\frac{d\mathbf H}{dt}
&= {}^B\frac{d\mathbf H}{dt}+\boldsymbol{\omega}\times\mathbf H \\
&= [I]\dot{\boldsymbol{\omega}}
    +\left({}^B\frac{d[I]}{dt}\right)\boldsymbol{\omega}
    +\boldsymbol{\omega}\times([I]\boldsymbol{\omega}) \\
&\quad +J_g\ddot{\gamma}\hat{\mathbf g}_g
    +I_{W_s}\dot{\Omega}\hat{\mathbf g}_s
    +I_{W_s}\Omega\dot{\gamma}\hat{\mathbf g}_t \\
&\quad +J_g\dot{\gamma}
    (\omega_t\hat{\mathbf g}_s-\omega_s\hat{\mathbf g}_t) \\
&\quad +I_{W_s}\Omega
    (\omega_g\hat{\mathbf g}_t-\omega_t\hat{\mathbf g}_g).
\end{aligned}
$$

Collecting the remaining terms by direction gives the same three coefficients:

$$
\begin{aligned}
c_s &= (J_s-J_t)\dot{\gamma}\omega_t+J_g\dot{\gamma}\omega_t
    +I_{W_s}\dot{\Omega}, \\
c_t &= (J_s-J_t)\dot{\gamma}\omega_s-J_g\dot{\gamma}\omega_s
    +I_{W_s}\Omega(\dot{\gamma}+\omega_g), \\
c_g &= J_g\ddot{\gamma}-I_{W_s}\Omega\omega_t.
\end{aligned}
$$

The first two recover the factors $J_s-J_t+J_g$ and $J_s-J_t-J_g$ from the EOM. So the compact route reproduces every remaining term.

**Locked gimbal: reaction-wheel limit.** Set both $\dot{\gamma}=0$ and $\ddot{\gamma}=0$. The body-frame inertia becomes constant, and the EOM reduces to

$$
\begin{aligned}
[I]\dot{\boldsymbol{\omega}}
&= \mathbf L-\boldsymbol{\omega}\times
    \left([I]\boldsymbol{\omega}+I_{W_s}\Omega\hat{\mathbf g}_s\right) \\
&\quad -I_{W_s}\dot{\Omega}\hat{\mathbf g}_s.
\end{aligned}
$$

This is the fixed-axis reaction-wheel case, including the inertia of the locked gimbal.

**Constant wheel speed: CMG operation.** Set $\dot{\Omega}=0$ while allowing the gimbal to move. The gimbal-acceleration and body-rate couplings remain. Holding $\Omega$ constant does not generally mean the spin motor supplies zero torque; we'll see why immediately below.

If both internal motions are locked, including $\Omega=0$, we recover the rigid-body Euler equation. With no external torque, $\mathbf L=\mathbf 0$, the full model conserves total inertial angular momentum while the motors redistribute it internally.


## 1.5 - Motor Torques

**<ins>Wheel Spin-Motor Torque</ins>**

The whole-system balance removes internal torques. To find what a motor actually supplies, isolate the part it acts on. Start with the wheel.

Let $u_s$ be the torque applied to the wheel in the positive spin direction. Let $\tau_{w_t}$ and $\tau_{w_g}$ be the transverse bearing reactions applied to it. Under the ideal joint model,

$$
\mathbf D_W={}^N\frac{d\mathbf H_W}{dt}
=u_s\hat{\mathbf g}_s+\tau_{w_t}\hat{\mathbf g}_t+\tau_{w_g}\hat{\mathbf g}_g.
$$

Project onto the spin axis. Orthogonality removes the transverse reactions, leaving the component we already derived:

$$
\begin{aligned}
u_s &= \hat{\mathbf g}_s^T\mathbf D_W=D_{W,s} \\
&= I_{W_s}(\dot{\omega}_s+\dot{\Omega}) \\
&= I_{W_s}(\alpha_s+\dot{\gamma}\omega_t+\dot{\Omega}).
\end{aligned}
$$

Restore $\alpha_s=\hat{\mathbf g}_s^T\dot{\boldsymbol{\omega}}$:

$$
\boxed{
u_s=I_{W_s}\left(\dot{\Omega}
    +\hat{\mathbf g}_s^T\dot{\boldsymbol{\omega}}+\dot{\gamma}\omega_t\right).
}
$$

So the motor responds to three contributions: relative spin acceleration, spacecraft angular acceleration along the spin axis, and the change in that body-rate projection as the gimbal turns. Even at constant $\Omega$, the last two can require a nonzero motor torque.

The other two projections are

$$
\tau_{w_t}=D_{W,t},\qquad \tau_{w_g}=D_{W,g}.
$$

These are real loads carried by the bearings, with equal-and-opposite reactions on the gimbal. They matter for the hardware even though they are not independently commanded motor inputs.


**<ins>Gimbal-Motor Torque</ins>**

For the gimbal motor, isolate the gimbal and wheel together. The wheel motor and wheel-bearing reactions are now internal to this pair, so they cancel.

Let $u_g$ be the torque applied by the spacecraft to the assembly in the positive gimbal direction. The other mounting reaction components are $\tau_{G_s}$ and $\tau_{G_t}$:

$$
\mathbf D_G+\mathbf D_W
=\tau_{G_s}\hat{\mathbf g}_s+\tau_{G_t}\hat{\mathbf g}_t+u_g\hat{\mathbf g}_g.
$$

Project onto the gimbal axis and substitute the two directional results:

$$
\begin{aligned}
u_g &= D_{G,g}+D_{W,g} \\
&= I_{G_g}(\alpha_g+\ddot{\gamma})
    +(I_{G_t}-I_{G_s})\omega_s\omega_t \\
&\quad +I_{W_t}(\alpha_g+\ddot{\gamma})
    +(I_{W_t}-I_{W_s})\omega_s\omega_t-I_{W_s}\Omega\omega_t \\
&= (I_{G_g}+I_{W_t})(\alpha_g+\ddot{\gamma}) \\
&\quad +[(I_{G_t}+I_{W_t})-(I_{G_s}+I_{W_s})]\omega_s\omega_t
    -I_{W_s}\Omega\omega_t \\
&= J_g(\alpha_g+\ddot{\gamma})
    +(J_t-J_s)\omega_s\omega_t-I_{W_s}\Omega\omega_t.
\end{aligned}
$$

With $\alpha_g=\hat{\mathbf g}_g^T\dot{\boldsymbol{\omega}}$, the motor torque is

$$
\boxed{
u_g=J_g\left(\hat{\mathbf g}_g^T\dot{\boldsymbol{\omega}}+\ddot{\gamma}\right)
    -(J_s-J_t)\omega_s\omega_t-I_{W_s}\Omega\omega_t.
}
$$

The first term accelerates the assembly about its gimbal axis. The second comes from unequal combined spin and transverse inertias. The last is the gyroscopic contribution from the wheel spin momentum.

The remaining mounting loads follow from the other two directions:

$$
\tau_{G_s}=D_{G,s}+D_{W,s},
\qquad
\tau_{G_t}=D_{G,t}+D_{W,t}.
$$

The spacecraft experiences the opposite assembly torque. This is how a motor acting inside the spacecraft can change the body's motion while total angular momentum remains conserved in the absence of external torque.

> **Reference wording correction:** the motor is only one component of the torque on an isolated subsystem. Bearing and mounting reactions also act externally on that subsystem, while all of these interaction torques are internal to the complete spacecraft-device system.


**<ins>Closing the Coupled Equations</ins>**

If $u_s$ and $u_g$ are the inputs, rearrange the two motor equations to expose the relative accelerations:

$$
\begin{aligned}
I_{W_s}\dot{\Omega}
&= u_s-I_{W_s}\alpha_s-I_{W_s}\dot{\gamma}\omega_t, \\
J_g\ddot{\gamma}
&= u_g-J_g\alpha_g+(J_s-J_t)\omega_s\omega_t+I_{W_s}\Omega\omega_t.
\end{aligned}
$$

Divide by the corresponding inertias and restore the acceleration projections:

$$
\boxed{
\dot{\Omega}=\frac{u_s}{I_{W_s}}
    -\hat{\mathbf g}_s^T\dot{\boldsymbol{\omega}}-\dot{\gamma}\omega_t,
}
$$

$$
\boxed{
\ddot{\gamma}
=\frac{u_g+(J_s-J_t)\omega_s\omega_t+I_{W_s}\Omega\omega_t}{J_g}
    -\hat{\mathbf g}_g^T\dot{\boldsymbol{\omega}}.
}
$$

Together with the three-component spacecraft EOM, these are five coupled equations for $\dot{\boldsymbol{\omega}}$, $\dot{\Omega}$, and $\ddot{\gamma}$. Evaluate the axes and $[I]$ at the current gimbal angle, using the state $(\boldsymbol{\omega},\Omega,\gamma,\dot{\gamma})$.

The spacecraft acceleration appears in both motor equations, and the internal accelerations appear in the spacecraft equation. That mutual dependence is the coupling we need to retain when solving the model.


## 1.6 - Single-VSCMG Numerical Simulation

# 2 - Multi-VSCMG Modeling

## 2.1 - Extending the Model to Multiple VSCMGs

## 2.2 - Numerical Implementation and Debugging

## 2.3 - 4-VSCMG Numerical Simulation

## 2.4 - Work-Energy Verification